# ScentMatch: fine-tuning multilingual-e5-small
Выберите в Colab GPU runtime, затем выполняйте ячейки сверху вниз. Архив проекта должен содержать `data/training`.

In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
archive_name = next(iter(uploaded))
project_dir = Path('/content/ScentMatch')
project_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_name) as archive:
    archive.extractall(project_dir)
print('Project:', project_dir)

In [ ]:
%pip install -q "sentence-transformers>=5,<6" "datasets>=3,<5" "accelerate>=1,<2"

In [ ]:
!python /content/ScentMatch/scripts/finetune_e5.py --project-dir /content/ScentMatch --output-dir /content/scentmatch-e5-finetuned

## Пересчёт embeddings и proxy-метрик
Сначала установим сам проект, затем построим отдельные embeddings fine-tuned модели.

In [ ]:
%cd /content/ScentMatch
%pip install -q -e ".[ml]"
!scentmatch-e5 build --model /content/scentmatch-e5-finetuned/final --artifact-dir artifacts/e5_finetuned --device cuda
!scentmatch-e5 evaluate --artifact-dir artifacts/e5_finetuned --device cuda --report data/interim/e5_finetuned_report.json --rankings artifacts/e5_finetuned/rankings.jsonl --review-csv data/interim/e5_finetuned_results_review.csv

In [ ]:
import shutil

model_zip = shutil.make_archive('/content/scentmatch-e5-finetuned', 'zip', '/content/scentmatch-e5-finetuned/final')
files.download(model_zip)